# Notebook 2 — Forward Process & Noise Distributions

This notebook:
- Implements `GaussianDiffusion`, `UniformDiffusion`, `LaplaceDiffusion`
- Verifies that all distributions are standardised (mean=0, std=1)
- Visualises the forward diffusion at multiple timesteps for all three
- Plots distribution shapes side-by-side

## 1. Setup (copy from Notebook 1)

In [ ]:
!pip install -q torch torchvision matplotlib numpy

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import os

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/diffusion_noise_project'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# Paste CONFIG and make_beta_schedule from Notebook 1, then:
CONFIG = {
    'dataset': 'MNIST', 'image_size': 28, 'channels': 1, 'batch_size': 128,
    'T': 1000, 'beta_start': 1e-4, 'beta_end': 0.02, 'schedule': 'linear',
    'num_epochs': 30, 'lr': 2e-4, 'optimizer': 'adamw', 'weight_decay': 1e-4,
    'num_workers': 2, 'model_channels': 64, 'channel_mults': (1, 2, 4),
    'log_every': 100, 'save_every': 5, 'seed': 42,
}

def make_beta_schedule(schedule, T, beta_start, beta_end):
    if schedule == 'linear':
        betas = torch.linspace(beta_start, beta_end, T)
    elif schedule == 'cosine':
        steps = T + 1
        x = torch.linspace(0, T, steps)
        alphas_cumprod = torch.cos(((x / T) + 0.008) / 1.008 * torch.pi / 2) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        betas = betas.clamp(0, 0.999)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), alphas_cumprod[:-1]])
    return {
        'betas': betas, 'alphas': alphas,
        'alphas_cumprod': alphas_cumprod,
        'alphas_cumprod_prev': alphas_cumprod_prev,
        'sqrt_alphas_cumprod': alphas_cumprod.sqrt(),
        'sqrt_one_minus_alphas_cumprod': (1 - alphas_cumprod).sqrt(),
    }

schedule = make_beta_schedule(CONFIG['schedule'], CONFIG['T'], CONFIG['beta_start'], CONFIG['beta_end'])

# Base class
class ForwardDiffusion:
    def __init__(self, schedule, device):
        self.device = device
        self.T = len(schedule['betas'])
        for k, v in schedule.items():
            setattr(self, k, v.to(device))
    def sample_noise(self, shape):
        raise NotImplementedError
    def sample_timestep(self, batch_size):
        return torch.randint(0, self.T, (batch_size,), device=self.device).long()
    def q_sample(self, x_0, t):
        eps = self.sample_noise(x_0.shape)
        sqrt_alpha_bar = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_one_minus_alpha_bar = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        x_t = sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * eps
        return x_t, eps
    def noise_stats(self, n_samples=100000):
        noise = self.sample_noise((n_samples, 1, 1, 1))
        return {'mean': noise.mean().item(), 'std': noise.std().item()}

print('Base classes loaded.')

## 2. Three Noise Distributions

All distributions are standardised to **mean=0, variance=1** so comparisons are fair.

| Distribution | Raw parameterisation | Standardised form |
|---|---|---|
| Gaussian | N(0, 1) | Already standard |
| Uniform | U(-a, a) | a = √3 so var = a²/3 = 1 |
| Laplace | Lap(0, b) | b = 1/√2 so var = 2b² = 1 |

In [ ]:
class GaussianDiffusion(ForwardDiffusion):
    """
    Standard Gaussian forward process: eps ~ N(0, I)
    This is the baseline used in DDPM and most diffusion models.
    """
    name = 'gaussian'

    def sample_noise(self, shape):
        return torch.randn(shape, device=self.device)


class UniformDiffusion(ForwardDiffusion):
    """
    Uniform forward process: eps ~ U(-√3, √3)
    Range √3 ≈ 1.732 chosen so variance = (2√3)²/12 = 1 (unit variance).
    """
    name = 'uniform'
    _a = 3 ** 0.5  # ≈ 1.732

    def sample_noise(self, shape):
        # U(-√3, √3)
        return torch.empty(shape, device=self.device).uniform_(-self._a, self._a)


class LaplaceDiffusion(ForwardDiffusion):
    """
    Laplace forward process: eps ~ Laplace(0, 1/√2)
    Scale b = 1/√2 chosen so variance = 2b² = 1 (unit variance).
    Laplace noise has heavier tails than Gaussian — kurtosis = 6 vs 3.
    """
    name = 'laplace'
    _b = 2 ** -0.5  # ≈ 0.707

    def sample_noise(self, shape):
        # Laplace(0, b): sample via inverse CDF
        u = torch.empty(shape, device=self.device).uniform_(-0.5, 0.5)
        return -self._b * u.sign() * torch.log(1 - 2 * u.abs())


print('All three diffusion classes defined.')

## 3. Verify Standardisation

In [ ]:
diffusions = {
    'Gaussian': GaussianDiffusion(schedule, DEVICE),
    'Uniform':  UniformDiffusion(schedule, DEVICE),
    'Laplace':  LaplaceDiffusion(schedule, DEVICE),
}

print(f"{'Distribution':<12} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print('-' * 55)
for name, diff in diffusions.items():
    noise = diff.sample_noise((500000, 1, 1, 1))
    print(f"{name:<12} {noise.mean().item():>10.4f} {noise.std().item():>10.4f} "
          f"{noise.min().item():>10.4f} {noise.max().item():>10.4f}")

## 4. Distribution Shape Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
colors = {'Gaussian': '#4C72B0', 'Uniform': '#DD8452', 'Laplace': '#55A868'}
x_range = np.linspace(-4, 4, 400)

for ax, (name, diff) in zip(axes, diffusions.items()):
    noise = diff.sample_noise((200000, 1, 1, 1)).cpu().numpy().flatten()
    ax.hist(noise, bins=150, density=True, alpha=0.7, color=colors[name], label='Sampled')

    # Theoretical PDFs
    if name == 'Gaussian':
        pdf = np.exp(-x_range**2 / 2) / np.sqrt(2 * np.pi)
        ax.plot(x_range, pdf, 'k-', lw=2, label='Theoretical')
    elif name == 'Uniform':
        a = 3**0.5
        pdf = np.where(np.abs(x_range) <= a, 1 / (2 * a), 0)
        ax.plot(x_range, pdf, 'k-', lw=2, label='Theoretical')
    elif name == 'Laplace':
        b = 2**-0.5
        pdf = np.exp(-np.abs(x_range) / b) / (2 * b)
        ax.plot(x_range, pdf, 'k-', lw=2, label='Theoretical')

    ax.set_title(f'{name} noise', fontsize=13)
    ax.set_xlabel('Value')
    ax.set_xlim(-4, 4)
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Density')
plt.suptitle('Noise distribution comparison (all standardised to mean=0, std=1)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'figures', 'noise_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Forward Process Visualisation
Show how each distribution corrupts the same image at t = 0, 250, 500, 750, 1000.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
test_dataset = torchvision.datasets.MNIST(
    root='/content/data', train=False, download=True, transform=transform
)

# Pick a clean sample
torch.manual_seed(CONFIG['seed'])
x_0 = test_dataset[7][0].unsqueeze(0).to(DEVICE)  # shape (1, 1, 28, 28)

timesteps = [0, 250, 500, 750, 999]
dist_names = list(diffusions.keys())

fig, axes = plt.subplots(len(dist_names), len(timesteps), figsize=(15, 9))

for row, (dist_name, diff) in enumerate(diffusions.items()):
    for col, t_val in enumerate(timesteps):
        if t_val == 0:
            img = x_0
        else:
            t_tensor = torch.tensor([t_val], device=DEVICE)
            img, _ = diff.q_sample(x_0, t_tensor)

        # Unnormalise
        img_display = (img.squeeze().cpu() * 0.5 + 0.5).clamp(0, 1).numpy()
        axes[row, col].imshow(img_display, cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')

        if row == 0:
            axes[row, col].set_title(f't = {t_val}', fontsize=12)
        if col == 0:
            axes[row, col].set_ylabel(dist_name, fontsize=12, rotation=90, labelpad=10)

plt.suptitle('Forward diffusion: same image, same beta schedule, different noise distributions',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'figures', 'forward_diffusion_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 6. Noise Power as a Function of t
Confirm that all three distributions add the same amount of variance at each step.

In [ ]:
# Sample a clean image and compute the variance of x_t over many noise samples
t_values = [50, 100, 250, 500, 750, 999]
n_repeats = 500

results = {name: [] for name in diffusions}

for t_val in t_values:
    t_tensor = torch.tensor([t_val], device=DEVICE)
    for name, diff in diffusions.items():
        variances = []
        for _ in range(n_repeats):
            x_t, _ = diff.q_sample(x_0, t_tensor)
            variances.append(x_t.var().item())
        results[name].append(np.mean(variances))

fig, ax = plt.subplots(figsize=(9, 5))
for name, vals in results.items():
    ax.plot(t_values, vals, marker='o', label=name, linewidth=2)

ax.set_xlabel('Timestep t')
ax.set_ylabel('Variance of x_t')
ax.set_title('Variance of noisy image vs timestep (should be near-identical across distributions)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'figures', 'noise_variance_vs_t.png'), dpi=150)
plt.show()

In [ ]:
print("Notebook 2 complete.")
print("Key takeaway: all three distributions add the same variance at each t.")
print("Differences are purely in the shape (tail behaviour) of the noise.")
print("Proceed to Notebook 3 to build the shared U-Net architecture.")